# Knowledge Graph Extraction dari Buku Teks

Ekstraksi Knowledge Graph **per-buku** dari PDF (Biologi, Fisika, Kimia Kelas XII).
Tahap ini hanya menghasilkan graf per-buku (konsep + relasi intra-buku) dan menyimpannya ke `outputs/*.json`.
Cross-book completion & ingest Neo4j ditangani pada notebook terpisah.


In [ ]:
import os, json, re
from pathlib import Path
from datetime import datetime

from google import genai
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

## Konfigurasi

In [ ]:
PROJECT_DIR = Path('/Users/fadrianyhoga/Documents/Skripsi')
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

BOOKS_CONFIG = [
    {"name": "Biologi Kelas XII", "pdf_siswa": "textbook/Biologi_BS_KLS_XII_Rev.pdf", "pdf_guru": "textbook/Biologi_BG_KLS_XII.pdf"},
    {"name": "Fisika Kelas XII",  "pdf_siswa": "textbook/Fisika_BS_KLS_XII.pdf",      "pdf_guru": "textbook/Fisika_BG_KLS_XII.pdf"},
    {"name": "Kimia Kelas XII",   "pdf_siswa": "textbook/Kimia_BS_KLS_XII.pdf",       "pdf_guru": "textbook/Kimia_BG_KLS_XII.pdf"},
]

# API key dibaca dari environment variable (jangan hard-code di notebook).
#   export GEMINI_API_KEY="..."   (atau set di .env / shell sebelum menjalankan)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY belum di-set. Jalankan: export GEMINI_API_KEY="..."')

LLM_MODEL_EXTRACT         = 'gemini-2.5-flash-lite'
CHUNK_SIZE, CHUNK_OVERLAP = 800, 200

# -- Intra-book relation types --
INTRA_RELATION_TYPES = [
    'MENDEFINISIKAN', 'MENYEBABKAN', 'MEMUNGKINKAN', 'MENGATUR',
    'BAGIAN_DARI', 'BERINTERAKSI_DENGAN', 'BERGANTUNG_PADA', 'MEMPENGARUHI',
]

client = genai.Client(api_key=GEMINI_API_KEY)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f'LLM Extract : {LLM_MODEL_EXTRACT}')
print(f'Chunk       : size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')
print(f'Books       : {[b["name"] for b in BOOKS_CONFIG]}')

## Ekstraksi KG dari PDF

### 1.1 Load PDF + Chunking

In [ ]:
all_books = {}

for cfg in BOOKS_CONFIG:
    name = cfg['name']
    docs   = SimpleDirectoryReader(input_files=[cfg['pdf_siswa']]).load_data()
    docs_g = SimpleDirectoryReader(input_files=[cfg['pdf_guru']]).load_data()
    parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    all_books[name] = {
        'documents': docs, 'documents_bg': docs_g,
        'nodes': parser.get_nodes_from_documents(docs),
        'nodes_bg': parser.get_nodes_from_documents(docs_g),
        'chapters': [], 'glossary': {}, 'materi_pokok_map': {},
        'chapter_nodes': {}, 'book_graph': None,
    }
    print(f'{name}: {len(all_books[name]["nodes"])} chunks (siswa), {len(all_books[name]["nodes_bg"])} chunks (guru)')

print(f'\nLoaded {len(all_books)} books.')

### 1.2 Extract TOC, Chapters, Glossary, Materi Pokok

In [ ]:
def extract_chapters_from_toc(toc_text):
    chapter_iter = list(re.finditer(
        r"(?i)(bab\s+(?:\d+|[IVXLCDM]+))\s+(.+?)\s*\.{2,}\s*(\d+)", toc_text))
    chapters = []
    for i, m in enumerate(chapter_iter):
        name = m.group(2).strip()
        page = int(m.group(3))
        if name.lower().startswith(('glosarium', 'indeks', 'daftar', 'rangkuman', 'asesmen', 'refleksi')):
            continue
        toc_start = m.end()
        toc_end = chapter_iter[i + 1].start() if i + 1 < len(chapter_iter) else len(toc_text)
        sub_matches = re.findall(
            r"(?m)^[ \t]*([A-Z])\.\s+(.+?)\s*\.{2,}\s*(\d+)",
            toc_text[toc_start:toc_end])
        subchapters = [{'label': l, 'name': n.strip(), 'page_start': int(p)} for l, n, p in sub_matches]
        chapters.append({'name': name, 'page_start': page, 'subchapters': subchapters})
    return sorted(chapters, key=lambda x: x['page_start'])


def extract_glossary(documents):
    glossary_text, in_glos = '', False
    for doc in documents[-30:]:
        low = doc.text.lower()
        if re.search(r'\bglosarium\b', low):
            in_glos = True
        if in_glos and re.search(r'\b(indeks|daftar pustaka|profil)\b', low):
            break
        if in_glos:
            glossary_text += doc.text + '\n'
    glossary = {}
    for line in glossary_text.split('\n'):
        line = line.strip()
        if ':' in line and len(line.split(':')) == 2:
            t, d = line.split(':', 1)
            glossary[t.strip()] = d.strip()
    return glossary


def extract_materi_pokok(documents_bg, chapters):
    mp_map = {ch['name']: [] for ch in chapters}
    mp_text = ''
    for doc in documents_bg:
        low = doc.text.lower()
        if any(x in low for x in ['materi pokok', 'pokok materi', 'skema pembelajaran']):
            mp_text += doc.text + '\n'
    lines = [l.strip() for l in mp_text.split('\n') if l.strip()]
    for ch_name in mp_map:
        for line in lines:
            if (len(line.split()) < 10 and line[0:1].isupper()
                    and not any(s in line for s in ['Peserta didik', 'Tujuan', 'Kegiatan'])):
                if line not in mp_map[ch_name]:
                    mp_map[ch_name].append(line)
    return mp_map


def roman_to_int(roman):
    vals = {'i': 1, 'v': 5, 'x': 10, 'l': 50, 'c': 100, 'd': 500, 'm': 1000}
    total, prev = 0, 0
    for ch in reversed(roman.lower()):
        v = vals.get(ch, 0)
        total += -v if v < prev else v
        prev = v
    return total


def get_page_number(node):
    label = node.metadata.get('page_label', '')
    if label.isdigit():
        return int(label)
    try:
        return roman_to_int(label)
    except Exception:
        return 0


for book_name, bd in all_books.items():
    print(f'\n--- {book_name} ---')
    # TOC
    toc_text = ''
    for doc in bd['documents'][:20]:
        low = doc.text.lower()
        if ('daftar isi' in low or 'bab' in low) and re.search(r'\.{3,}\s*\d+', doc.text):
            toc_text += doc.text + '\n'
    chapters = extract_chapters_from_toc(toc_text)
    # Enrich prev/next
    for i, ch in enumerate(chapters):
        ch['previous'] = chapters[i-1]['name'] if i > 0 else None
        ch['next'] = chapters[i+1]['name'] if i < len(chapters)-1 else None
        ch['next_page'] = chapters[i+1]['page_start'] if i < len(chapters)-1 else 10000
    print(f'  Chapters: {len(chapters)}')
    # Glossary
    glossary = extract_glossary(bd['documents'])
    print(f'  Glossary: {len(glossary)} terms')
    # Materi pokok
    mp_map = extract_materi_pokok(bd['documents_bg'], chapters)
    # Assign nodes to chapters
    chapter_nodes = {}
    for node in bd['nodes']:
        page = get_page_number(node)
        for ch in chapters:
            if ch['page_start'] <= page < ch['next_page']:
                chapter_nodes.setdefault(ch['name'], []).append(node)
                break
    for ch_name, nodes in chapter_nodes.items():
        print(f'    {ch_name}: {len(nodes)} nodes')

    bd['chapters'] = chapters
    bd['glossary'] = glossary
    bd['materi_pokok_map'] = mp_map
    bd['chapter_nodes'] = chapter_nodes

print('\nPreprocessing complete.')

### 1.3 Extraction Function

In [ ]:
def extract_triplets(text_chunk, grade, chapter, subchapters=None,
                     previous_chapter=None, next_chapter=None,
                     glossary=None, materi_pokok=None):
    sub_section = ''
    if subchapters:
        sub_list = '\n'.join(f'  {s["label"]}. {s["name"]}' for s in subchapters)
        sub_section = f'SUBCHAPTER:\n{sub_list}\nGunakan nama subchapter persis untuk field "name" di "subtopics".'

    mp_section = ''
    if materi_pokok:
        mp_list = '\n'.join(f'  {i+1}. {m}' for i, m in enumerate(materi_pokok[:25]))
        mp_section = f'MATERI POKOK (BUKU GURU):\n{mp_list}'

    glos_section = ''
    if glossary:
        glos_items = '\n'.join(f'  {t}: {d[:100]}' for t, d in list(glossary.items())[:25])
        glos_section = f'GLOSARIUM:\n{glos_items}'

    rel_types = '|'.join(INTRA_RELATION_TYPES)
    prompt = f"""Kamu membangun knowledge graph dari buku teks Kelas XII.
HIERARKI: Kelas -> Bab -> Subchapter -> Konsep

Buku: {grade} | Bab: {chapter}
Bab sebelumnya: {previous_chapter or "Tidak ada"} | Bab berikutnya: {next_chapter or "Tidak ada"}

{sub_section}
{mp_section}
{glos_section}

OUTPUT FORMAT (STRICT JSON):
{{
  "chapter_summary": "Ringkasan 3-5 kalimat.",
  "subtopics": [{{
    "name": "...",
    "concepts": [{{
      "name": "...", "description": "Deskripsi 1-2 kalimat.",
      "glossary_validated": true/false, "materi_pokok_ref": "...",
      "relations": [{{
        "type": "{rel_types}",
        "target": "...", "description": "..."
      }}]
    }}]
  }}],
  "chapter_relations": [{{
    "type": "PRASYARAT|MEMPERSIAPKAN", "target_chapter": "...",
    "description": "...", "concept_links": [{{"source_concept": "...", "target_concept": "...", "explanation": "..."}}]
  }}]
}}

Semua output Bahasa Indonesia. Setiap relations WAJIB punya field target non-kosong.

Teks:
{text_chunk}"""

    response = client.models.generate_content(model=LLM_MODEL_EXTRACT, contents=prompt)
    text = response.text.replace('```json', '').replace('```', '').strip()
    parsed = json.loads(text)

    # Bersihkan relations: target & type wajib non-kosong.
    for sub in parsed.get('subtopics', []):
        for c in sub.get('concepts', []):
            c['relations'] = [
                {'type': r['type'].strip(), 'target': r['target'].strip(),
                 'description': r.get('description', '').strip()}
                for r in (c.get('relations') or [])
                if (r.get('type') or '').strip() and (r.get('target') or '').strip()
            ]
    return parsed

### 1.4 Run Extraction

In [ ]:
for book_name, bd in all_books.items():
    print(f'\n=== {book_name} ===')
    chapters = bd['chapters']
    chapter_nodes = bd['chapter_nodes']

    book_graph = {'grade': book_name, 'chapters': []}
    for ch in chapters:
        ch_name = ch['name']
        ch_text = '\n\n'.join(n.text for n in chapter_nodes.get(ch_name, []))
        if not ch_text.strip():
            print(f'  SKIP {ch_name}: no text')
            continue
        try:
            result = extract_triplets(
                ch_text, book_name, ch_name,
                subchapters=ch.get('subchapters', []),
                previous_chapter=ch['previous'], next_chapter=ch['next'],
                glossary=bd['glossary'],
                materi_pokok=bd['materi_pokok_map'].get(ch_name, []),
            )
            ch_result = {
                'chapter': ch_name,
                'chapter_summary': result.get('chapter_summary', ''),
                'previous': ch['previous'], 'next': ch['next'],
                'subchapters': [s['name'] for s in ch.get('subchapters', [])],
                'subtopics': result.get('subtopics', []),
                'chapter_relations': result.get('chapter_relations', []),
            }
            concept_n = sum(len(st.get('concepts', [])) for st in ch_result['subtopics'])
            book_graph['chapters'].append(ch_result)
            print(f'  {ch_name}: {len(ch_result["subtopics"])} subtopics, {concept_n} concepts')
        except Exception as e:
            print(f'  ERROR {ch_name}: {str(e)[:120]}')

    bd['book_graph'] = book_graph
    print(f'  Done: {len(book_graph["chapters"])} chapters')

print('\nExtraction complete.')

### 1.5 Save

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
saved = []
for book_name, bd in all_books.items():
    bg = bd.get('book_graph')
    if not bg:
        print(f'WARNING: no book_graph for {book_name}')
        continue
    fname = f'{book_name}_{timestamp}.json'
    fpath = OUTPUTS_DIR / fname
    fpath.write_text(json.dumps(bg, ensure_ascii=False, indent=2), encoding='utf-8')
    saved.append(fname)
    print(f'Saved: {fname}')

print(f'\n{len(saved)} books saved to {OUTPUTS_DIR}')